In [1]:
import re
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

In [2]:
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog barks at the quick brown fox.",
    "A lazy dog sleeps under the old tree.",
    "The fox runs into the dark forest.",
    "The brown dog chases the quick fox."
]

In [3]:
tokenized = []

In [4]:
for sentence in corpus:
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z\s]', '', sentence)
    words_in_sentence = sentence.split()
    tokenized.append(words_in_sentence)

In [5]:
words = []

In [6]:
for sent in tokenized:
    for w in sent:
        if w not in words:
            words.append(w)

In [7]:
words = sorted(words)

word_to_index = {}
index_to_word = {}

In [8]:
for i in range(len(words)):
    word_to_index[words[i]] = i
    index_to_word[i] = words[i]

vocab_size = len(words)

In [9]:
print("Vocabulary:", word_to_index)

Vocabulary: {'a': 0, 'at': 1, 'barks': 2, 'brown': 3, 'chases': 4, 'dark': 5, 'dog': 6, 'forest': 7, 'fox': 8, 'into': 9, 'jumps': 10, 'lazy': 11, 'old': 12, 'over': 13, 'quick': 14, 'runs': 15, 'sleeps': 16, 'the': 17, 'tree': 18, 'under': 19}


In [10]:
X = []
y = []
window = 2


In [11]:
for sent in tokenized:
    for i in range(window, len(sent) - window):

        context = sent[i-2:i] + sent[i+1:i+3]
        target = sent[i]

        context_indices = []

        for w in context:
            context_indices.append(word_to_index[w])

        X.append(context_indices)
        y.append(word_to_index[target])

X = np.array(X)
y = np.array(y)

In [12]:
print("Sample Context:", X[0])
print("Sample Target:", y[0])

# 5. Build CBOW model
model = Sequential([
    Embedding(vocab_size, 10),
    GlobalAveragePooling1D(),
    Dense(vocab_size, activation='softmax')
])

# 6. Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 7. Train model
history = model.fit(X, y, epochs=500, batch_size=8, verbose=0)

print("Training Loss:", history.history['loss'][-1])

# 8. Display word embeddings
embeddings = model.layers[0].get_weights()[0]

print("Word Embeddings:")

for word, index in word_to_index.items():
    print(word, embeddings[index])

# 9. Predict target word
context = ["the", "quick", "fox", "jumps"]

test = []

for w in context:
    test.append(word_to_index[w])

test = np.array([test])

prediction = model.predict(test, verbose=0)

predicted_index = np.argmax(prediction)

print("Predicted Target Word:", index_to_word[predicted_index])

Sample Context: [17 14  8 10]
Sample Target: 3
Training Loss: 0.6788690686225891
Word Embeddings:
a [-0.37901953  0.8572959   0.6751843  -0.49273124 -0.6098826   0.6589559
  0.63828754  0.64337003  0.539047   -0.7319528 ]
at [-0.18757111 -1.2923025  -0.84120625 -0.06369063  0.6008799   0.69662195
  0.6428543  -0.6345972  -0.5350165  -0.01355784]
barks [-0.16143435 -0.9467887  -0.73122424 -0.7382309   0.736621    0.29688534
  0.84026194 -0.08940319 -0.71791184  0.7877239 ]
brown [-0.61212397  0.15277177  0.0466164  -0.01388092  0.71647364 -0.0160313
 -0.17280218 -0.17350395 -1.1835045   0.12291775]
chases [-1.0437927   0.51947486 -0.0197266  -1.1012919   0.02469425  0.901457
  1.020487    1.3313138  -0.31990528  0.02432503]
dark [-0.7723405   0.7103884  -1.0165997  -0.7242958  -0.44110185  0.68453395
 -0.09926619  1.0397867   0.220497    1.1486484 ]
dog [-0.14675629 -1.5899234  -0.5538242  -0.25023967 -0.56354845 -0.03095327
  1.1955237  -0.5969014  -0.62102807  1.0753266 ]
forest [-0.6

In [13]:
# Predict target words
test_contexts = [
    ["the", "quick", "fox", "jumps"],
    ["the", "brown", "jumps", "over"],
    ["a", "lazy", "sleeps", "under"],
    ["the", "dog", "at", "the"]
]

for context in test_contexts:

    test = []

    for w in context:
        test.append(word_to_index[w])

    test = np.array([test])

    prediction = model.predict(test, verbose=0)

    predicted_index = np.argmax(prediction)

    print("Context:", context)
    print("Predicted Word:", index_to_word[predicted_index])
    print()

Context: ['the', 'quick', 'fox', 'jumps']
Predicted Word: brown

Context: ['the', 'brown', 'jumps', 'over']
Predicted Word: fox

Context: ['a', 'lazy', 'sleeps', 'under']
Predicted Word: dog

Context: ['the', 'dog', 'at', 'the']
Predicted Word: barks

